# **Trabajo Práctico Final**
---
## ***Procesamiento del Lenguaje Natural - TUIA***

| Autora | Legajo |
| --- | --- |
Rizzotto, María Camila | R-4676/1

# Ejercicio 1 - RAG

## Carga del Repositorio Pradera

Para tener acceso a las 3 fuentes de datos del juego que me fue asignado

In [1]:
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.colab import auth
import io
import os

# Autenticar
auth.authenticate_user()
drive_service = build('drive', 'v3')

# ID de la carpeta en Drive con los datos del juego PRADERA
FOLDER_ID = '1bQazMZnk73rRaG5btORykstehZABNf-C'
DESTINO_LOCAL = '/content/datos_pradera'

# Crear carpeta local si no existe
os.makedirs(DESTINO_LOCAL, exist_ok=True)

# Función recursiva para descargar carpetas con subcarpetas
def descargar_carpeta(folder_id, ruta_destino):
    query = f"'{folder_id}' in parents and trashed=false"
    resultados = drive_service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = resultados.get('files', [])

    for item in items:
        nombre = item['name']
        file_id = item['id']
        tipo = item['mimeType']

        if tipo == 'application/vnd.google-apps.folder':
            # Es una subcarpeta
            nueva_ruta = os.path.join(ruta_destino, nombre)
            os.makedirs(nueva_ruta, exist_ok=True)
            descargar_carpeta(file_id, nueva_ruta)
        else:
            # Es un archivo, lo descargamos
            request = drive_service.files().get_media(fileId=file_id)
            fh = io.FileIO(os.path.join(ruta_destino, nombre), 'wb')
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()

# Ejecutar la descarga
descargar_carpeta(FOLDER_ID, DESTINO_LOCAL)
print(f"✅ Carpeta descargada con subcarpetas en: {DESTINO_LOCAL}")

✅ Carpeta descargada con subcarpetas en: /content/datos_pradera


##Base de Datos vectorial


Para este ejercicio, voy a utilizar Chroma con LangChain como base de datos vectorial. En cuanto al modelo de embedding, usaré 'distiluse-base-multilingual-cased-v1' ya que yo tengo textos en varios idiomas. Y para el split de texto, vuelvo a utilizar spaCy como en mi anterior trabajo, pero agrupando de a 3 oraciones

In [2]:
%%capture
!pip install sentence-transformers chromadb langchain spacy
!python -m spacy download xx_ent_wiki_sm
!pip install -U langchain-community

In [3]:
from pathlib import Path
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import spacy

### Text Split

In [4]:
# Cargamos modelo SpaCy multilingue para la segmentación
nlp = spacy.load("xx_ent_wiki_sm")
nlp.add_pipe("sentencizer") #detecta los límites de oración basado en signos de puntuación

# Leemos todos los archivos .txt de la carpeta 'informacion'
folder_path  = Path("/content/datos_pradera/informacion")
documentos = []
documentos_fragmentados = []
for archivo in folder_path.glob("*.txt"):
    with open(archivo, encoding="utf-8") as f:
        texto = f.read()
        doc = Document(page_content=texto, metadata={"fuente": archivo.name})
        documentos.append(doc)
        doc_spacy = nlp(texto)

        # Agrupamos cada 3 oraciones como un fragmento temático básico
        oraciones = list(doc_spacy.sents)
        for i in range(0, len(oraciones), 3):
            fragmento = " ".join([str(s) for s in oraciones[i:i+3]])
            if fragmento.strip():
                documentos_fragmentados.append(Document(page_content=fragmento, metadata={"fuente": archivo.name}))

print(f"Documentos originales: {len(documentos)}")
print(f"Fragmentos generados: {len(documentos_fragmentados)}")

Documentos originales: 15
Fragmentos generados: 1884


### Cargo modelo de embeddings

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/distiluse-base-multilingual-cased-v1"
)

/tmp/ipython-input-5-1452574728.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

2_Dense/pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

### Base de Datos

In [6]:
# Creamos base de datos vectorial
db = Chroma.from_documents(
    documents=documentos_fragmentados,
    embedding=embedding_model,
    persist_directory="./chroma_pradera"
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Interfaz

In [7]:
#Cuando encontramos una respuesta que no esté en español, la vamos a traducir
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00


In [8]:
from deep_translator import GoogleTranslator

def traducir_a_espanol(texto):
    try:
        return GoogleTranslator(source='auto', target='es').translate(texto)
    except Exception as e:
        print("Error al traducir:", e)
        return texto

In [46]:
from langchain.schema import Document

def busqueda_semantica(query, k=3):
    """Función que devuelve objetos Document compatibles con el retriever."""
    resultados = db.similarity_search(query, k=k)
    documentos = [
        Document(
            page_content=r.page_content,
            metadata={"fuente": r.metadata["fuente"]}
        )
        for r in resultados
    ]
    return documentos

### Consultas

In [11]:
import os
os.environ["CHROMA_TELEMETRY"] = "FALSE" #Para que no salga el warning de chroma y ensucie las consultas

In [49]:
query = "Cual es la mecánica del juego?"
resultados = busqueda_semantica(query, k=4)

print("Query:", query, "\n")

for i, doc in enumerate(resultados, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    fuente = doc.metadata.get("fuente", "Desconocida")

    print(f"Respuesta {i}:\n{traduccion}")
    print(f"- Fuente: {fuente}\n")

Query: Cual es la mecánica del juego? 

Respuesta 1:
Esto no quiere decir que ese motor de juego no sea notable; Meadow presenta algunas mecánicas muy inteligentes que equilibran el juego. Por ejemplo, las dos avenidas de puntuación, los cuadros y las bonificaciones del tablero de fogatas, están bien ponderadas entre sí dadas la frecuencia con la que se juega cada uno, y la puntuación generalmente no es demasiado oscuro. El uso de cuatro mazos separados, cada uno con su propia distribución de tipos de tarjetas y símbolos, para alimentar el grupo de tarjetas compartidas asegura una disponibilidad abundante de tipos de tarjetas y símbolos para todos los jugadores y evita que los jugadores sigan con éxito una estrategia molesta de recursos de asfixia.
- Fuente: foro_reviews.txt

Respuesta 2:
El juego en sí es bastante sencillo, ya que la mayoría de las veces, quieres jugar cartas que muestren símbolos que necesitarás para que jueguen otras cartas: puede parecer un poco abstracta, pero nue

In [50]:
query = "how do i win"
resultados = busqueda_semantica(query, k=4)

print("Query:", query, "\n")

for i, doc in enumerate(resultados, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    fuente = doc.metadata.get("fuente", "Desconocida")

    print(f"Respuesta {i}:\n{traduccion}")
    print("   (Respuesta original:",doc.page_content,")") #evidencia lo que fue traducido
    print(f"- Fuente: {fuente}\n")

Query: how do i win 

Respuesta 1:
Puedes ganar Meadow si puedes mostrar los puntos más de victoria después de un número determinado de rondas. 

¿Cómo funciona? 

Antes del primer juego, debes armar algunos soportes para tarjetas de cartón.
   (Respuesta original: Gewinnen kann man Meadow, wenn man nach einer vorgegebenen Rundenanzahl die meisten Siegpunkte vorweisen kann. 

Wie läuft das ab? 

Vor dem ersten Spiel muss man ersteinmal ein paar Kartenhalter aus Pappe zusammenbauen. )
- Fuente: foro_reviews.txt

Respuesta 2:
Lo máximo que he logrado llegar hasta ahora es como 5-7. Era la única forma en que logré ganar ese juego porque la persona contra la que estaba jugando obtuve todos los puntos de bonificación, y no obtuve ninguno, ¡así que comencé a concentrarme en las carreteras y tuve mucha suerte con las cartas que aparecieron! 
Sí, las carreteras son una muy buena estrategia cuando los demás no lo hacen ;-)

Hola, tengo una tarjeta con un solo requisito.
   (Respuesta original: 

In [20]:
query = "jugadores maximos permitidos"
resultados = busqueda_semantica(query, k=4)

print("Query:", query, "\n")

for i, doc in enumerate(resultados, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    fuente = doc.metadata.get("fuente", "Desconocida")

    print(f"Respuesta {i}:\n{traduccion}")
    print(f"- Fuente: {fuente}\n")

Query: jugadores maximos permitidos  

Respuesta 1: ¿No hay límites de mano con tarjetas? ¿No hay límites de tokens de carretera?: ¿No es este juego ni límite de tamaño de la mano ni límite de tokens de carretera por jugador?
- Fuente: foro_general.txt  

Respuesta 2: Fuera de mi cabeza, la única restricción que conozco (juego base) es que no puedes colocar nada encima de una carta con el ícono Ungulado (Cabeza de los ciervos). Por supuesto, no todos tendrán los ungulados en juego (ya que dependen de los sobres de apertura), por lo que incluso esa restricción puede no aplicarse ...

2 preguntas menores: una tarjeta de paisaje nunca puede tener más de otra tarjeta (descubrimiento) colocada encima, ¿verdad? 

¿Existe una razón específica por la cual las fichas de carretera deben colocarse bajo paisajes?
- Fuente: foro_reglas.txt  

Respuesta 3: A través del juego podrás obtener Max 24 (en el juego 1, 2, 3 jugadores) o 16 (en el juego de 4 jugadores) Tokens de carretera si usas todos tus 

##Acceso a los Datos Estadísticos

Carga de información en df de Pandas

In [2]:
import pandas as pd

#Importamos datos de estadísticas en un dataframe
estadisticas_pradera = pd.read_csv("/content/datos_pradera/estadisticas/meadow_stats.csv", sep=",")
estadisticas_pradera

,Estadistica,Valor
0,Avg. Rating,7.719
1,No. of Ratings,"12,201"
2,Std. Deviation,1.22
3,Weight,2.25 / 5
4,Comments,"1,861"
5,Fans,"1,164"
6,Page Views,"983,188"
7,Overall Rank,209
8,Strategy Rank,158
9,Family Rank,28


### Procesamiento de información

In [3]:
#Recopilado información de importancia
#drop filas irrelevantes (como aquellas estadísticas muy relativas a la página y no al juego)
estadisticas_relevantes_pradera = estadisticas_pradera.drop([2,4,6,11,13,14,15,17,18])

In [17]:
# Limpieza de datos
estadisticas_relevantes_pradera["Estadistica"] = estadisticas_relevantes_pradera["Estadistica"].str.strip()
estadisticas_relevantes_pradera["Valor"] = estadisticas_relevantes_pradera["Valor"].astype(str).str.strip()

# Creamos un diccionario resumen
estadisticas = dict(zip(estadisticas_relevantes_pradera["Estadistica"], estadisticas_relevantes_pradera["Valor"]))

# Construimos el string para el LLM
info_llm = "\n".join([f"{k}: {v}" for k, v in estadisticas.items()])
print(info_llm)

Avg. Rating: 7.719
No. of Ratings: 12,201
Weight: 2.25 / 5
Fans: 1,164
Overall Rank: 209
Strategy Rank: 158
Family Rank: 28
All Time Plays: 50,530
Own: 23,140
Wishlist: 4,854


### Uso de un modelo de lenguaje

Como esta tarea es simple, voy a usar un modelo liviano: Mistral-7B-Instruct GGUF

In [7]:
!pip install llama-cpp-python --upgrade --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 MB 8.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00


In [20]:
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

#Descargar modelo
model_path = hf_hub_download(
    repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF",
    filename="mistral-7b-instruct-v0.2.Q4_K_M.gguf"
)

#Cargarlo con Llama
llm = Llama(
    model_path=model_path,
    n_ctx=2048,
    n_threads=4,
    verbose=False
)

mistral-7b-instruct-v0.2.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


In [57]:
def extraer_filtro_estadistico(pregunta_usuario, info_llm):
    prompt = f"""<|system|>
Eres un modelo experto en datos estadísticos de juegos de mesa.
Tu tarea es responder preguntas sobre el juego devolviendo el **nombre exacto del campo** que contiene la respuesta.
Debes elegir solo uno, el más adecuado, **entre los campos listados en la información**.

Información disponible:
{info_llm}

Formato esperado:
'<nombre exacto del campo>'

Ejemplos:

Usuario: ¿Cuántas personas agregaron el juego a su wishlist?
Respuesta:
'Wishlist'

Usuario: ¿Cuál es el puntaje promedio del juego?
Respuesta:
'Avg. Rating'

Usuario: ¿Cuántos fans tiene el juego?
Respuesta:
'Fans'

Usuario: ¿Cuántas veces se jugó el juego en total?
Respuesta:
'All Time Plays'

Usuario: {pregunta_usuario}
Respuesta:
"""

    respuesta = llm(prompt, max_tokens=200, stop=["Usuario:"])
    return respuesta["choices"][0]["text"].strip()


In [58]:
respuesta = extraer_filtro_estadistico("¿Cuántas veces se jugó el juego en total?", info_llm)
print(respuesta)

'All Time Plays'


In [80]:
respuesta = extraer_filtro_estadistico("en que ranking esta el juego", info_llm)
print(respuesta)

'Overall Rank'


In [82]:
respuesta = extraer_filtro_estadistico("cuantos fans tiene meadow", info_llm)
print(respuesta)

'Fans'


### Interfaz

In [92]:
def filtrar_set_estadistico(filtro):
  """Esta funcion filtra el set de datos según el resultado del LLM usado y obtiene una respuesta cceder mediante"""
  filtro = filtro.replace("'", "").replace('"', '')
  valor_estadistico = estadisticas_relevantes_pradera[estadisticas_relevantes_pradera["Estadistica"] == filtro]
  return valor_estadistico

def obtener_valor_estadistico(consulta):
  """Se integran todas funciones realizadas: pasamos una consulta en lenguaje natural, un LLM obtiene un filtro y se trae la respuesta del set de"""
  filtro = extraer_filtro_estadistico(consulta, info_llm)
  valor_estadistico = filtrar_set_estadistico(filtro)
  return valor_estadistico

In [93]:
obtener_valor_estadistico("que ranking de estrategia ocupa el juego?")

,Estadistica,Valor
8,Strategy Rank,158


In [94]:
obtener_valor_estadistico("que puntaje promedio tiene meadow")

,Estadistica,Valor
0,Avg. Rating,7.719


##Base de Datos de Grafos - Neo4j + Cypher

### Creación Base de Datos de Grafo

In [2]:
!pip install py2neo pandas sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.7 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

Importamos archivo CSV -> Dataframe Pandas

In [20]:
from py2neo import Graph, Node, Relationship
import pandas as pd
#Importamos datos de relaciones en un dataframe
relaciones_pradera = pd.read_csv("/content/datos_pradera/relaciones/relaciones_juego.csv", sep=",")

In [7]:
relaciones_pradera.head()

,SUJETO1,RELACION,SUJETO2
0,Meadow,NOMBRE_ALTERNATIVO,Łąka
1,Meadow,NOMBRE_ALTERNATIVO,Livada
2,Meadow,NOMBRE_ALTERNATIVO,Meadow Im Reich der Natur
3,Meadow,NOMBRE_ALTERNATIVO,Na louce
4,Meadow,NOMBRE_ALTERNATIVO,Pradera


Conexión a mi instancia de Neo4j

In [3]:
graph = Graph("neo4j+s://3ccc91f3.databases.neo4j.io", auth=("neo4j", "wWhDo2bDgYcsoaLV9ZueOLnqGDpdH9qCB4qM0WIaHKg"))
# Verificación de conexión
graph.run("RETURN 'Conexión exitosa con Neo4j!' AS mensaje").data()

[{'mensaje': 'Conexión exitosa con Neo4j!'}]

Inserción de datos (creación BBDD)

In [22]:
# Inserto datos en grafo
for _, row in relaciones_pradera.iterrows():
    s1 = str(row["SUJETO1"]).strip()
    rel = str(row["RELACION"]).strip()
    s2 = str(row["SUJETO2"]).strip()

    # Crear nodos si no existen
    nodo1 = Node("Entidad", nombre=s1)
    nodo2 = Node("Entidad", nombre=s2)
    graph.merge(nodo1, "Entidad", "nombre")
    graph.merge(nodo2, "Entidad", "nombre")

    # Crear relación
    relacion = Relationship(nodo1, rel.upper().replace(" ", "_"), nodo2)
    graph.merge(relacion)

###Implementación modelo de lenguaje: consulta lenguaje natural -> consulta Cypher

Voy a utilizar un modelo de Sentence Transformer para obtener rápida y fácilmente una interfaz de consulta traduciendo del lenguaje natural -> consultas cypher, mediante una base de preguntas y consultas

In [4]:
from sentence_transformers import SentenceTransformer, util

# Cargo un modelo preentrenado
modelo = SentenceTransformer("all-MiniLM-L6-v2")

# Lista de preguntas frecuentes y sus respectivas queries Cypher
base_preguntas = [
    "¿Quién ilustró Meadow?",
    "¿Qué mecanismos tiene el juego?",
    "¿Qué categorías tiene el juego?",
    "¿Quién diseñó el juego?",
    "¿Con qué editoriales se publicó Meadow?",
    "¿Qué personas trabajaron juntas?",
    "¿Qué otros nombres tiene el juego Meadow?"
]

base_consultas = [
    "MATCH (j:Entidad {nombre: 'Meadow'})-[:ILUSTRADOR]->(i) RETURN i.nombre",
    "MATCH (j:Entidad {nombre: 'Meadow'})-[:MECANISMO]->(m) RETURN m.nombre",
    "MATCH (j:Entidad {nombre: 'Meadow'})-[:CATEGORIA]->(c) RETURN c.nombre",
    "MATCH (j:Entidad {nombre: 'Meadow'})-[:DISEÑADOR]->(d) RETURN d.nombre",
    "MATCH (j:Entidad {nombre: 'Meadow'})-[:EDITORIAL]->(e) RETURN e.nombre",
    "MATCH (a:Entidad)-[:RELACION_INTERNA]->(b:Entidad) RETURN a.nombre, b.nombre",
    "MATCH (j:Entidad {nombre: 'Meadow'})-[:NOMBRE_ALTERNATIVO]->(n) RETURN n.nombre"
]

# Calculo embeddings base
embeddings_base = modelo.encode(base_preguntas, convert_to_tensor=True)

#----------------INTERFAZ-----------------

def graph_search(pregunta):
    # Embedding de la pregunta nueva
    emb_pregunta = modelo.encode(pregunta, convert_to_tensor=True)

    # Comparación de similitud con las preguntas conocidas
    similitudes = util.pytorch_cos_sim(emb_pregunta, embeddings_base)[0]
    idx_mejor = similitudes.argmax().item()
    confianza = similitudes[idx_mejor].item()

    # Tomo un umbral de confianza
    if confianza < 0.60:
        return {"cypher": None, "respuesta": "No entiendo la consulta. ¿Podés reformular?"}

    # Ejecutamos la consulta Cypher seleccionada
    cypher_query = base_consultas[idx_mejor]
    try:
        resultado = graph.run(cypher_query).data()
    except Exception as e:
        return {"cypher": cypher_query, "error": str(e)}

    return {
        "cypher": cypher_query,
        "respuesta": resultado
    }

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Ejemplo de uso y consultas

In [6]:
#Query
pregunta = "¿Quién hizo los dibujos del juego?"

#Implemento interfaz
resultado = graph_search(pregunta)

#Imprimo resultados
print("Query: ", pregunta)
print("\nConsulta Cypher generada:")
print(resultado["cypher"])
print("\nRespuesta:")
print(resultado["respuesta"])

Query:  ¿Quién hizo los dibujos del juego?

Consulta Cypher generada:
MATCH (j:Entidad {nombre: 'Meadow'})-[:DISEÑADOR]->(d) RETURN d.nombre

Respuesta:
[{'d.nombre': 'Klemens Kalicki'}]


## Clasificador de Intención

En este punto voy a comparar modelos que reciben una consulta y la categoricen entre la fuente de datos que pueda llegar a responder esa pregunta entre estadísticas,
información y relaciones.
Por ejemplo:
- ¿Cómo gano en el ajedrez? -> Información
- ¿Quién trabajó para el ta-te-ti? -> Relaciones
- ¿Qué puntaje tienen las damas? -> Estadística

###Modelo del TP anterior

Había realizado 2: uno con regresión logística y otro de redes neuronales. El último fue el que mejores resultados me dio, así que lo voy a traer a este colab:

In [116]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
from sentence_transformers import SentenceTransformer

#Cargo modelo guardado llamado modelo_NN.keras
modelo_NN = tf.keras.models.load_model('modelo_NN.keras')

#Modelo de embedding que había usado
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

#Predicciones
def predecir_categoria_nn(frases, mostrar_resultados=True):
    frases = [f.lower() for f in frases]
    embeddings = model.encode(frases)
    predicciones = modelo_NN.predict(embeddings)
    etiquetas = np.argmax(predicciones, axis=1)
    mapeo_inverso = {0: "estadísticas", 1: "información", 2: "relaciones"}
    categorias_predichas = [mapeo_inverso[p] for p in etiquetas]

    #Mostrar resultados
    if mostrar_resultados:
      for frase, categoria in zip(frases, categorias_predichas):
          print(f"'{frase}' → {categoria}")
    return categorias_predichas

In [98]:
#Probamos algunas predicciones
predecir_categoria_nn(["quien creo el juego","cuantos usuarios juegan","como desempato"])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step
'quien creo el juego' → relaciones
'cuantos usuarios juegan' → estadísticas
'como desempato' → relaciones


Acá se nota que se confundió en la última respuesta que debía ser 'información' y no 'relaciones'. Veremos si ahora un modelo con LLM tiene mejor performance

### Basado en LLM con Few-Shot Prompting

Voy a reciclar el modelo de llama que usé en el punto de acceso a datos estadísticos, pero ajustando el prompt a los nuevo requerimientos para ver qué tal funciona.

In [99]:
def clasificar_intencion_llm(pregunta_usuario):
    prompt = f"""<|system|>
Sos un asistente experto en el juego Meadow. Tu tarea es clasificar las consultas de los usuarios según la fuente de datos más adecuada para responderla.

Las posibles fuentes son:
- Información: para preguntas sobre reglas, estrategias, mecánica del juego, contenido textual general.
- Relaciones: para preguntas sobre personas que colaboraron, conexiones entre autores, ilustradores, editoriales, etc.
- Estadísticas: para preguntas sobre puntaje, cantidad de votos, rankings, dificultad, cantidad de jugadas, etc.

Formato de respuesta:
Información / Relaciones / Estadísticas

Ejemplos:

Usuario: ¿Cómo gano en el juego?
Respuesta: Información

Usuario: ¿Quién trabajó en la ilustración?
Respuesta: Relaciones

Usuario: ¿Qué puntaje obtuvo en el ranking?
Respuesta: Estadísticas

Usuario: ¿Cuántos fans tiene el juego Pradera?
Respuesta: Estadísticas

Usuario: ¿Qué mecánicas utiliza el juego Pradera?
Respuesta: Información

Usuario: ¿Quién diseñó el juego Pradera?
Respuesta: Relaciones

Usuario: ¿Cuándo fue lanzado el juego Pradera?
Respuesta: Relaciones

Usuario: {pregunta_usuario}
Respuesta:"""

    respuesta = llm(prompt, max_tokens=10, stop=["Usuario:"])
    return respuesta["choices"][0]["text"].strip()

Vamos a hacerle las mismas preguntas que le hicimos al otro modelo

In [101]:
clasificar_intencion_llm('quien creo el juego')

'Relaciones'

In [102]:
clasificar_intencion_llm('cuantos usuarios juegan')

'Estadísticas'

In [103]:
clasificar_intencion_llm('como desempato')

'Información'

Buena respuesta en esta última query, que es la que el modelo NN no pudo predecir. Para realizar una buena comparación, vamos a hacer un pequeño set de preguntas para testear y calculamos los accuracy de ambos modelos

In [123]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Set de validación extendido
test_set = [
    ("¿Cuál es la puntuación promedio del juego?", "estadísticas"),
    ("¿Cómo se juega Pradera?", "información"),
    ("quién diseñó el juego?", "relaciones"),
    ("Cuántas personas tienen el juego en su colección?", "estadísticas"),
    ("cuales son las mecánicas del juego?", "información"),
    ("¿Quién ilustró el juego?", "relaciones"),
    ("¿Qué dificultad tiene el juego?", "estadísticas"),
    ("¿Cuántos votos recibió?", "estadísticas"),
    ("cuantas personas pueden jugar a la vez", "información"),
    ("en que año se lanzo pradera", "relaciones"),
    ("¿Quién escribió las reglas?", "relaciones"),
    ("que ranking ocupa", "estadísticas"),
    ("Cuántos comentarios tiene?", "estadísticas"),
    ("¿cómo funciona la mecánica de colocación de trabajadores?", "información"),
    ("¿qué diseñadores colaboraron entre sí?", "relaciones"),
    ("Qué editorial lo publicó?", "relaciones"),
    ("¿qué tipo de estrategia se puede usar?", "información"),
    ("Cuántos fans tiene el juego?", "estadísticas"),
    ("¿Qué recursos incluye el manual?", "información"),
    ("¿Quién editó la versión original?", "relaciones")
]

# Separar inputs y etiquetas reales
preguntas = [x[0] for x in test_set]
etiquetas_reales = [x[1] for x in test_set]

# Realizamos predicciones en la red neuronal
predicciones_nn = predecir_categoria_nn(preguntas, False)

# Métricas red neuronal
print("Accuracy red neuronal:", accuracy_score(etiquetas_reales, predicciones_nn))

# Evaluar LLM sobre el mismo set
def evaluar_llm_clasificador(preguntas, etiquetas_reales):
    pred_llm = [clasificar_intencion_llm(p).strip().lower() for p in preguntas]
    etiquetas_reales = [e.lower() for e in etiquetas_reales]
    acc = accuracy_score(etiquetas_reales, pred_llm)
    return acc
#Métricas LLM
accuracy_llm = evaluar_llm_clasificador(preguntas, etiquetas_reales)
print("LLM Accuracy:", accuracy_llm)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Accuracy red neuronal: 0.7
LLM Accuracy: 0.95


Con estas métricas se puede comparar y ver claramente que el clasificador basado en un LLM es superior a la red neuronal que hicimos anteriormente. Con un buen prompt se logró que realice gran cantidad de clasificaciones correctas, aunque la red neuronal también logra un accuracy bastante alto. Este último modelo podría mejorarse ampliando su dataset de entrenamiento, pero con las métricas obtenidas se nota que para la tarea de este punto el modelo basado en LLM es mejor, más eficaz y práctico.

## Pipeline de Recuperación (Retrieval)

### Para consultas en BBDD semánticas

####Búsqueda híbrida: semántica + por palabras clave

In [23]:
!pip install rank_bm25

In [25]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.schema import Document
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')

# 1. BM25 Retriever con mis fragmentos ya guardados en 'documentos_fragmentados'
def tokenize(doc):
    return word_tokenize(doc.page_content.lower())

bm25_retriever = BM25Retriever.from_documents(documentos_fragmentados)
bm25_retriever.k = 5  #podemos ajustar

# 2. Wrapper para mi búsqueda semántica existente (busqueda_semantica definida como interfaz de BBDD vectorial)
class CustomSemanticRetriever:
    def __init__(self, k=5):
        self.k = k

    def get_relevant_documents(self, query):
        resultados = busqueda_semantica(query, k=self.k)
        return [Document(page_content=r.page_content, metadata=r.metadata) for r in resultados]

semantic_retriever = CustomSemanticRetriever(k=5)

# 3. Retriever Híbrido (BM25 + Semántica)
def retriever_hibrido(query, k_total=5):
    """
    Búsqueda híbrida: mezcla BM25 y búsqueda semántica.
    Combina resultados, priorizando relevancia cruzada y eliminando duplicados.
    """
    # Recupera resultados
    resultados_bm25 = bm25_retriever.get_relevant_documents(query)
    resultados_semanticos = busqueda_semantica(query, k=k_total)

    # Combina ambos (elimina duplicados por contenido)
    texto_visto = set()
    resultados_combinados = []
    for doc in resultados_bm25 + resultados_semanticos:
        if doc.page_content not in texto_visto:
            resultados_combinados.append(doc)
            texto_visto.add(doc.page_content)

        if len(resultados_combinados) >= k_total:
            break

    return resultados_combinados

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Ejemplo de uso:

In [57]:
query = "como gano el juego?"
resultados = retriever_hibrido(query, 4)

for i, r in enumerate(resultados):
    print(f"\nResultado {i+1}:\n{r.page_content[:]}")


Resultado 1:
Gracias como siempre compañero. 
2 A mi me tiene encandilado. No deja de ser un juego de draft y colecciones, pero el sistema matricial del quadropolis eleva el nivel de exigencia y la competición por las bonificaciones el nivel de interacción.

Resultado 2:
Avanzarán guiados por la pasión, 

la curiosidad por el mundo, una mente inquisitiva y el deseo de descubrir los misterios de la naturaleza y conver-
tirse en el observador más habilidoso. Ganará quien consiga más puntos observando los diferentes tipos de ani-
males, plantas y paisajes, así como reuniendo recuerdos durante su viaje. La competición continúa en la hoguera, 
donde los jugadores compiten por cumplir con los objetivos de sus aventuras.

Resultado 3:
Y es que Pradera propone a los jugadores tomar el papel de exploradores que registrarán los hallazgos que logren visualizar durante su travesía, como pueden ser bellos ejemplares, tanto de flora como de fauna, diversos tipos de terreno, paisajes embriagadores y

#### Re-ranking

In [51]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from sentence_transformers import SentenceTransformer

#Uso el mismo modelo multilingüe que ya me funcionó
rerank_model = SentenceTransformer("sentence-transformers/distiluse-base-multilingual-cased-v1")

def rerank_resultados(query, documentos, model, top_k=5):
    """
    Reordena los documentos según su similitud semántica con la consulta.
    """
    # Embedding del query
    embedding_query = model.encode([query])

    # Embeddings de los documentos
    textos = [doc.page_content for doc in documentos]
    embeddings_docs = model.encode(textos)

    # Similaridades
    similitudes = cosine_similarity(embedding_query, embeddings_docs)[0]

    # Ordenamos según similitud
    indices_ordenados = np.argsort(similitudes)[::-1]
    documentos_ordenados = [documentos[i] for i in indices_ordenados[:top_k]]

    return documentos_ordenados

def buscar_fragmentos_hibrido_rerank(query, k=5):
    # 1. Buscar documentos con método híbrido
    resultados = retriever_hibrido(query, k_total=k*2)  # recuperamos más para rerankear

    # 2. Aplicar rerank
    resultados_rankeados = rerank_resultados(query, resultados, model=rerank_model, top_k=k)

    return resultados_rankeados

In [56]:
resultados_finales = buscar_fragmentos_hibrido_rerank("como gano el juego?",4)
for i, doc in enumerate(resultados_finales, 1):
    traduccion = traducir_a_espanol(doc.page_content)
    print(f"\nResultado {i}:\n{traduccion}\nFuente: {doc.metadata['fuente']}")


Resultado 1:
Puedes ganar Meadow si puedes mostrar los puntos más de victoria después de un número determinado de rondas. 

¿Cómo funciona? 

Antes del primer juego, debes armar algunos soportes para tarjetas de cartón.
Fuente: foro_reviews.txt

Resultado 2:
La mayoría de los puntos gana. Es algo que es como un gastador. 

Realmente me gusta este juego mientras juego, pero siempre extraño los juegos que no tienen la tensión de un objetivo.
Fuente: foro_variantes.txt

Resultado 3:
No puedo llegar a exposiciones, etc. Entonces, ¿cómo alguien como yo tiene sus manos en las promociones? 
Boantziggy
@Boardziggy
Tengo muchas ganas de obtener este juego, pero soy un complemento. Aquí en el Reino Unido es costoso obtener las promociones y, no sé, me quedo molesto cuando hay más contenido, pero está fuera de su alcance.
Fuente: foro_general.txt

Resultado 4:
Siempre acabo las partidas con la sensacion de que me falta una ronda y por qué se acaba tan pronto… xD
3 Gran reseña, como siempre. Está